In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Amazon Reviews EDA") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/30 16:06:32 WARN Utils: Your hostname, Shreyashs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.178 instead (on interface en0)
26/05/30 16:06:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/30 16:06:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/30 16:06:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
reviews = spark.read.parquet("/Volumes/T7 Shield/data-project/parquet/reviews.parquet")
metadata = spark.read.parquet("/Volumes/T7 Shield/data-project/parquet/metadata.parquet")

In [8]:
reviews.printSchema()
metadata.printSchema()

root
 |-- rating: double (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)

root
 |-- parent_asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- rating_number: long (nullable = true)
 |-- store: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- price: double (nullable = true)



In [5]:
metadata.count()

1587421

In [14]:
reviews.show()

+------+--------------------+--------------------+-----------+--------------------+-------------+------------+-----------------+
|rating|               title|                text|parent_asin|             user_id|    timestamp|helpful_vote|verified_purchase|
+------+--------------------+--------------------+-----------+--------------------+-------------+------------+-----------------+
|   5.0|        Crazy comfy!|Not gonna lie- th...| B0BGFR76CF|AFKZENTNBQ7A7V7UX...|1677321053520|           8|             true|
|   5.0|          Excellent!|  I love it. Pretty!| B00NXQLFQQ|AFKZENTNBQ7A7V7UX...|1523093771676|           0|             true|
|   5.0|    Best saddle pads|Huge fan of B Ver...| B0957WLR63|AGGZ357AO26RQZVRL...|1653526919105|           0|             true|
|   5.0|  Perfect repair kit|I have a great We...| B00IET8S80|AGGZ357AO26RQZVRL...|1627330911189|           0|             true|
|   5.0|         Works great|This was great fo...| B01C2SW7XA|AGGZ357AO26RQZVRL...|1617831811976|

In [15]:
metadata.show()

+-----------+--------------------+-----------------+--------------+-------------+------------------+--------------------+-----+
|parent_asin|               title|    main_category|average_rating|rating_number|             store|          categories|price|
+-----------+--------------------+-----------------+--------------+-------------+------------------+--------------------+-----+
| B01HDXC8AG|Sure-Grip Zombie ...|             NULL|           4.5|           84|         Sure-Grip|[Sports & Outdoor...| 55.0|
| B07R5BQ4YD|USGI Wet Weather ...|             NULL|           4.2|           14|              USGI|[Sports & Outdoor...| NULL|
| B003K8GZ7G|NHL San Jose Shar...|             NULL|           4.5|           28|            Aminco|[Sports & Outdoor...|18.99|
| B08GC4GBWB|Bont Skates - Pro...|Sports & Outdoors|           4.2|           36|              Bont|[Sports & Outdoor...|209.0|
| B07BYV947H|Team Golf Alamaba...|Sports & Outdoors|           5.0|            3|         Team Golf|[Spo

In [16]:
# REVIEWS TABLE (Fact Table)
#         |
#         | parent_asin
#         |
# METADATA TABLE (Dimension Table)

In [14]:
from pyspark.sql.functions import col, broadcast

# Reviews table
reviews_clean = reviews.select(
    "parent_asin",
    col("title").alias("review_title"),
    "text",
    "rating",
    "user_id",
    "timestamp",
    "helpful_vote",
    "verified_purchase"
)

# Metadata table
metadata_clean = metadata.select(
    "parent_asin",
    col("title").alias("product_title"),
    "main_category",
    "average_rating",
    "rating_number",
    "store",
    "categories",
    "price"
)

# Join without broadcast
joined = reviews_clean.join(
    metadata_clean,
    on="parent_asin",
    how="inner"
)

# Remove duplicate rows if needed
joined = joined.dropDuplicates()



In [15]:
# Save as parquet
joined.write \
    .mode("overwrite") \
    .parquet("/Volumes/T7 Shield/data-project/parquet/amazon_reviews_nlp.parquet")

In [19]:
from pyspark.sql.functions import broadcast

joined = reviews.join(
    broadcast(metadata),
    "parent_asin",
    "inner"
)

In [20]:
metadata.select(
    "main_category",
    "categories"
).show(
    20,
    truncate=False,
    vertical=True
)


-RECORD 0---------------------------------------------------------------------------------------------------------------------------------------
 main_category | NULL                                                                                                                           
 categories    | [Sports & Outdoors, Sports, Skates, Skateboards & Scooters, Skateboarding, Skateboard Parts, Wheels]                           
-RECORD 1---------------------------------------------------------------------------------------------------------------------------------------
 main_category | NULL                                                                                                                           
 categories    | [Sports & Outdoors, Sports, Boating & Sailing, Boating, Dry Bags]                                                              
-RECORD 2-------------------------------------------------------------------------------------------------------------------------

In [21]:
from pyspark.sql.functions import explode

category_df = metadata.select(
    "parent_asin",
    "main_category",
    explode("categories").alias("subcategory")
)

category_df.show(20, False)

+-----------+-----------------+------------------------------+
|parent_asin|main_category    |subcategory                   |
+-----------+-----------------+------------------------------+
|B01HDXC8AG |NULL             |Sports & Outdoors             |
|B01HDXC8AG |NULL             |Sports                        |
|B01HDXC8AG |NULL             |Skates, Skateboards & Scooters|
|B01HDXC8AG |NULL             |Skateboarding                 |
|B01HDXC8AG |NULL             |Skateboard Parts              |
|B01HDXC8AG |NULL             |Wheels                        |
|B07R5BQ4YD |NULL             |Sports & Outdoors             |
|B07R5BQ4YD |NULL             |Sports                        |
|B07R5BQ4YD |NULL             |Boating & Sailing             |
|B07R5BQ4YD |NULL             |Boating                       |
|B07R5BQ4YD |NULL             |Dry Bags                      |
|B003K8GZ7G |NULL             |Sports & Outdoors             |
|B003K8GZ7G |NULL             |Fan Shop                

In [23]:
# number of products per catagory

from pyspark.sql import functions as F

metadata.groupBy(
    "main_category"
).agg(
    F.sum(F.lit(1)).alias("products")
).orderBy(
    F.desc("products")
).show(20, False)

+-------------------------+--------+
|main_category            |products|
+-------------------------+--------+
|Sports & Outdoors        |870267  |
|AMAZON FASHION           |259564  |
|NULL                     |249954  |
|Amazon Home              |74995   |
|Automotive               |45629   |
|Tools & Home Improvement |25580   |
|Toys & Games             |12477   |
|Industrial & Scientific  |9081    |
|Pet Supplies             |7999    |
|Cell Phones & Accessories|6905    |
|Health & Personal Care   |6894    |
|All Electronics          |3235    |
|Office Products          |2588    |
|Camera & Photo           |1899    |
|Arts, Crafts & Sewing    |1883    |
|All Beauty               |1309    |
|Sports Collectibles      |1230    |
|Books                    |1214    |
|Computers                |971     |
|Grocery                  |867     |
+-------------------------+--------+
only showing top 20 rows


In [24]:
# top products
metadata.groupBy(
    "store"
).agg(
    F.sum(F.lit(1)).alias("products")
).orderBy(
    F.desc("products")
).show(20, False)

+---------------------------+--------+
|store                      |products|
+---------------------------+--------+
|NULL                       |35109   |
|adidas                     |16327   |
|WinCraft                   |15917   |
|New Era                    |11442   |
|'47                        |10858   |
|Outerstuff                 |10026   |
|Nike                       |8791    |
|Reebok                     |7986    |
|Majestic                   |7772    |
|FOCO                       |7551    |
|Generic                    |6373    |
|Under Armour               |5777    |
|SHIMANO                    |5258    |
|Northwest                  |5210    |
|Rico Industries            |4968    |
|Easton                     |4844    |
|WILSON                     |4243    |
|VF LSG                     |4207    |
|College Flags & Banners Co.|4113    |
|Elite Fan Shop             |4010    |
+---------------------------+--------+
only showing top 20 rows


In [25]:
# price kpis
metadata.select(
    F.avg("price").alias("avg_price"),
    F.max("price").alias("max_price"),
    F.min("price").alias("min_price")
).show()

+-----------------+---------+---------+
|        avg_price|max_price|min_price|
+-----------------+---------+---------+
|57.36083746256646| 21999.98|      0.0|
+-----------------+---------+---------+



In [26]:
# avg price kpis 
metadata.select(
    F.avg("average_rating"),
    F.max("average_rating"),
    F.min("average_rating")
).show()

+-------------------+-------------------+-------------------+
|avg(average_rating)|max(average_rating)|min(average_rating)|
+-------------------+-------------------+-------------------+
|  4.192648453056566|                5.0|                1.0|
+-------------------+-------------------+-------------------+



In [27]:
# review length kpis 
reviews.withColumn(
    "review_len",
    F.length("text")
).select(
    F.avg("review_len"),
    F.max("review_len")
).show()

[Stage 37:===================================>                    (10 + 6) / 16]

+------------------+---------------+
|   avg(review_len)|max(review_len)|
+------------------+---------------+
|183.27970459046796|           1000|
+------------------+---------------+



In [28]:
# review by year 
reviews.withColumn(
    "review_year",
    F.year(
        F.from_unixtime(
            F.col("timestamp")/1000
        )
    )
).groupBy(
    "review_year"
).agg(
    F.sum(F.lit(1)).alias("reviews")
).orderBy(
    "review_year"
).show()

[Stage 40:================================>                        (9 + 7) / 16]

+-----------+-------+
|review_year|reviews|
+-----------+-------+
|       2000|     24|
|       2001|     70|
|       2002|     65|
|       2003|    112|
|       2004|    293|
|       2005|   1283|
|       2006|   4051|
|       2007|  14428|
|       2008|  22553|
|       2009|  34399|
|       2010|  60793|
|       2011| 122927|
|       2012| 221029|
|       2013| 590680|
|       2014| 979673|
|       2015|1514200|
|       2016|1861396|
|       2017|1827544|
|       2018|1868974|
|       2019|2399675|
+-----------+-------+
only showing top 20 rows


In [29]:
# top reviewed products
reviews.groupBy(
    "parent_asin"
).agg(
    F.sum(F.lit(1)).alias("review_count")
).orderBy(
    F.desc("review_count")
).show(20)

[Stage 43:===================================>                    (10 + 6) / 16]

+-----------+------------+
|parent_asin|review_count|
+-----------+------------+
| B00NWXLQD2|       30369|
| B07BQRWTDJ|       23638|
| B0C5RBPW2Y|       20298|
| B01L6RE7Z4|       16590|
| B09MJKJYLQ|       15679|
| B0B7J8Y581|       14565|
| B0C5XW2T2N|       14478|
| B09LW2KHPM|       14029|
| B0BBFB48YQ|       13986|
| B0BTNZ41Y7|       13856|
| B00NPLSZF8|       13541|
| B08D9FWVJD|       13017|
| B00BGO0Q9O|       12744|
| B0BJH866HJ|       11966|
| B01M0D52OK|       11471|
| B094DXJZPN|       11184|
| B09SS1NK1B|       10725|
| B07TDPP7LQ|       10264|
| B0BX5QFWQN|        9592|
| B0BV6JM41B|        9491|
+-----------+------------+
only showing top 20 rows


In [34]:
# catagory performance 
joined.groupBy(
    "main_category"
).agg(
    F.avg("rating").alias("avg_rating")
).orderBy(
    F.desc("avg_rating")
).show()

[Stage 50:=======>                                                  (1 + 7) / 8]

+--------------------+------------------+
|       main_category|        avg_rating|
+--------------------+------------------+
|Magazine Subscrip...| 4.973684210526316|
|      Amazon Fire TV|4.8076923076923075|
|            Handmade| 4.626506024096385|
|       Digital Music| 4.589762076423937|
|         Video Games| 4.539104177151485|
|Arts, Crafts & Se...| 4.480130244949308|
|   Collectible Coins|            4.4375|
|       Entertainment| 4.382978723404255|
| Sports Collectibles| 4.333063209076175|
|         Movies & TV| 4.322115384615385|
|     Office Products| 4.292837494341725|
|      AMAZON FASHION|  4.27690355085911|
|        Buy a Kindle|4.2604813664596275|
|               Books| 4.255728206478799|
|         Amazon Home| 4.253987503880782|
|Industrial & Scie...| 4.244067188714209|
|Tools & Home Impr...| 4.235177865612648|
| Musical Instruments| 4.231000752445448|
|           Computers| 4.230515916575192|
|          Automotive| 4.228130794060385|
+--------------------+------------

In [35]:
# catagory popularity
joined.groupBy(
    "main_category"
).agg(
    F.sum(F.lit(1)).alias("reviews")
).orderBy(
    F.desc("reviews")
).show()

[Stage 59:====================================>                     (5 + 3) / 8]

+--------------------+--------+
|       main_category| reviews|
+--------------------+--------+
|   Sports & Outdoors|12518746|
|      AMAZON FASHION| 2080060|
|                NULL| 1835081|
|         Amazon Home| 1133792|
|Tools & Home Impr...|  501446|
|          Automotive|  430937|
|Health & Personal...|  235765|
|        Toys & Games|  205388|
|Cell Phones & Acc...|  182966|
|Industrial & Scie...|  150561|
|        Pet Supplies|   71232|
|     All Electronics|   65616|
|      Camera & Photo|   39384|
|     Office Products|   28719|
|             Grocery|   21249|
|          All Beauty|   20387|
|Arts, Crafts & Se...|   13513|
|           Computers|   11843|
|         Video Games|    9935|
|                Baby|    9859|
+--------------------+--------+
only showing top 20 rows


In [36]:
# rating and price corealtion 
joined.select(
    F.corr(
        "price",
        "rating"
    ).alias("price_rating_corr")
).show()

[Stage 68:=====================>                                    (3 + 5) / 8]

+--------------------+
|   price_rating_corr|
+--------------------+
|-0.01042973262313...|
+--------------------+



In [37]:
# best products
product_stats.filter(
    F.col("review_count") > 100
).orderBy(
    F.desc("avg_review_rating")
).show(20, False)

[Stage 77:=============================>                            (4 + 4) / 8]

+-----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+------------------+------------+
|parent_asin|title                                                                                                                                                                                                |main_category    |avg_review_rating |review_count|
+-----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+------------------+------------+
|B00F9GIS4K |Tough-Grid 750lb Forest Camo Paracord/Parachute Cord - Genuine Mil Spec Type IV 750lb Paracord Used by The US Military (MIl-C-5040-H) - 100% Nylon - 200Ft. - Forest Camo                            |Spo

In [38]:
# worst products 
product_stats.filter(
    F.col("review_count") > 100
).orderBy(
    F.asc("avg_review_rating")
).show(20, False)

[Stage 82:=====================>                                    (3 + 5) / 8]

+-----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+------------------+------------+
|parent_asin|title                                                                                                                                                                                               |main_category    |avg_review_rating |review_count|
+-----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+------------------+------------+
|B0BM44NNDB |FFOK Golf Training Mat for Swing Detection Batting, Premium Golf Impact Mat, Path Feedback Golf Practice Mats, Advanced Golf Hitting Mat for Indoor/Outdoor, Golf Training Aid Equipment, Green     |Amazon 

In [39]:
# NLP : Text cleaning 
from pyspark.sql.functions import *

In [40]:
reviews_nlp = reviews.withColumn(
    "clean_text",
    lower(col("text"))
)

reviews_nlp = reviews_nlp.withColumn(
    "clean_text",
    regexp_replace("clean_text", "[^a-zA-Z ]", "")
)

reviews_nlp.select(
    "text",
    "clean_text"
).show(5, False)

[Stage 83:>                                                         (0 + 1) / 1]

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [42]:
# review length insignts 
# 1 star -> short angry reviews?
# 5 star -> detailed reviews?
reviews_nlp = reviews_nlp.withColumn(
    "review_length",
    length("clean_text")
)

In [43]:
reviews_nlp.groupBy("rating").agg(
    avg("review_length").alias("avg_length")
).orderBy("rating").show()

+------+------------------+
|rating|        avg_length|
+------+------------------+
|   1.0|198.52834310475288|
|   2.0|233.88535110347112|
|   3.0|228.21295921009033|
|   4.0|223.12714232954883|
|   5.0|154.74094858125844|
+------+------------------+



In [44]:
# common words
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="clean_text",
    outputCol="words"
)

words_df = tokenizer.transform(reviews_nlp)

In [45]:
# explode
words = words_df.select(
    explode("words").alias("word")
)

In [46]:
# remove tiny words
words = words.filter(
    length("word") > 3
)

In [47]:
# top words
words.groupBy("word") \
    .agg(sum(lit(1)).alias("freq")) \
    .orderBy(desc("freq")) \
    .show(50, False)

[Stage 87:====================================================>   (15 + 1) / 16]

+-------+-------+
|word   |freq   |
+-------+-------+
|this   |9223111|
|with   |5670908|
|that   |5020266|
|great  |4971563|
|have   |4428210|
|very   |4117236|
|they   |3525688|
|good   |3068561|
|these  |2815863|
|like   |2542712|
|well   |2464309|
|just   |2362155|
|them   |2313709|
|would  |2163585|
|easy   |2162660|
|when   |2125012|
|product|2100296|
|love   |2053292|
|will   |1944092|
|quality|1728338|
|more   |1617881|
|from   |1611308|
|than   |1578161|
|nice   |1496086|
|really |1488225|
|bought |1457098|
|only   |1453391|
|your   |1442721|
|used   |1416027|
|time   |1386751|
|works  |1346638|
|what   |1334565|
|price  |1318597|
|little |1308178|
|perfect|1303309|
|after  |1277613|
|water  |1275812|
|about  |1247184|
|work   |1231130|
|dont   |1223646|
|much   |1133953|
|also   |1130878|
|size   |1100524|
|made   |1094133|
|other  |1089692|
|because|1086622|
|bike   |1065650|
|back   |1063775|
|were   |1063505|
|light  |1036478|
+-------+-------+
only showing top 50 rows


In [48]:
# positive vs negative words
positive = reviews_nlp.filter(
    col("rating") >= 4
)

In [49]:
negative = reviews_nlp.filter(
    col("rating") <= 2
)

In [52]:
# top positive words
positive_words = positive.select(
    explode(split(col("clean_text"), " ")).alias("word")
)

positive_words = positive_words.filter(
    length("word") > 3
)

positive_words = positive_words.filter(
    col("word") != ""
)

positive_words.groupBy("word") \
    .count() \
    .orderBy(desc("count")) \
    .show(30, False)

[Stage 96:=================================================>      (14 + 2) / 16]

+-------+-------+
|word   |count  |
+-------+-------+
|this   |7034880|
|great  |4631175|
|with   |4440617|
|that   |3621337|
|have   |3327070|
|very   |3313626|
|they   |2577555|
|good   |2576643|
|these  |2251385|
|well   |2132263|
|easy   |2002526|
|love   |1952175|
|like   |1915135|
|them   |1758837|
|just   |1742842|
|product|1593339|
|when   |1536612|
|would  |1460519|
|will   |1403679|
|quality|1401833|
|nice   |1307530|
|perfect|1256180|
|more   |1253977|
|works  |1198831|
|really |1185975|
|than   |1185255|
|from   |1141964|
|price  |1137335|
|bought |1109644|
|little |1082322|
+-------+-------+
only showing top 30 rows


In [53]:
negative.select(
    explode(split(col("clean_text"), " ")).alias("word")
).filter(
    (length("word") > 3) & (col("word") != "")
).groupBy("word") \
 .count() \
 .orderBy(desc("count")) \
 .show(30, False)

[Stage 99:====================================================>   (15 + 1) / 16]

+-------+-------+
|word   |count  |
+-------+-------+
|this   |1554823|
|that   |873411 |
|with   |793123 |
|have   |696071 |
|they   |615473 |
|very   |542982 |
|would  |461622 |
|after  |415121 |
|just   |396405 |
|when   |386359 |
|these  |383029 |
|product|380798 |
|like   |377157 |
|them   |368517 |
|will   |342007 |
|from   |327329 |
|only   |312138 |
|time   |300649 |
|dont   |291498 |
|first  |281729 |
|even   |269145 |
|good   |252687 |
|back   |252380 |
|work   |249122 |
|your   |248886 |
|bought |248511 |
|used   |247813 |
|were   |240794 |
|broke  |233097 |
|because|231734 |
+-------+-------+
only showing top 30 rows


In [55]:
# sentiments from rating 
reviews_nlp = reviews_nlp.withColumn(
    "sentiment",
    when(col("rating") >= 4, "positive")
    .when(col("rating") == 3, "neutral")
    .otherwise("negative")
)

In [56]:
reviews_nlp.groupBy("sentiment") \
    .agg(sum(lit(1)).alias("reviews")) \
    .show()

+---------+--------+
|sentiment| reviews|
+---------+--------+
| positive|15500168|
| negative| 2770091|
|  neutral| 1324911|
+---------+--------+



In [57]:
# product sentiments
joined = reviews_nlp.join(
    metadata,
    "parent_asin"
)

In [58]:
joined.groupBy(
    "main_category",
    "sentiment"
).agg(
    sum(lit(1)).alias("reviews")
).show(50, False)

[Stage 109:==========================================>              (6 + 2) / 8]

+----------------------------+---------+-------+
|main_category               |sentiment|reviews|
+----------------------------+---------+-------+
|Toys & Games                |negative |35426  |
|Amazon Home                 |positive |903561 |
|Musical Instruments         |negative |902    |
|Car Electronics             |positive |272    |
|Car Electronics             |neutral  |26     |
|Digital Music               |negative |72     |
|GPS & Navigation            |negative |247    |
|Unique Finds                |positive |45     |
|Books                       |negative |477    |
|Camera & Photo              |positive |31557  |
|Automotive                  |positive |342181 |
|All Electronics             |positive |48414  |
|Computers                   |positive |9442   |
|Health & Personal Care      |negative |47445  |
|Industrial & Scientific     |neutral  |9799   |
|Collectibles & Fine Art     |positive |510    |
|Arts, Crafts & Sewing       |neutral  |628    |
|Appliances         

In [59]:
# Find complements 
one_star = reviews_nlp.filter(
    col("rating") == 1
)

In [60]:
# top compliment words
one_star.select(
    explode(
        split("clean_text"," ")
    ).alias("word")
).filter(
    length("word") > 4
).groupBy("word") \
 .agg(sum(lit(1)).alias("freq")) \
 .orderBy(desc("freq")) \
 .show(50, False)

[Stage 114:===================================================>   (15 + 1) / 16]

+------------+------+
|word        |freq  |
+------------+------+
|would       |286347|
|product     |277381|
|after       |274526|
|these       |241671|
|first       |194160|
|money       |172210|
|broke       |171227|
|bought      |164848|
|return      |148912|
|quality     |139924|
|because     |138583|
|didnt       |131091|
|about       |129832|
|there       |121722|
|disappointed|114656|
|water       |113644|
|doesnt      |109530|
|small       |108747|
|cheap       |104927|
|waste       |104550|
|could       |104455|
|other       |97448 |
|really      |92590 |
|which       |90179 |
|tried       |89240 |
|great       |87982 |
|never       |86311 |
|plastic     |86274 |
|months      |85355 |
|received    |85292 |
|recommend   |79645 |
|again       |78868 |
|before      |77122 |
|amazon      |76008 |
|ordered     |75743 |
|light       |74236 |
|better      |73718 |
|right       |73485 |
|thing       |72883 |
|using       |71361 |
|still       |70941 |
|purchased   |70353 |
|times    

In [70]:
# clear previous spark
spark.catalog.clearCache()
spark.stop()

In [71]:
# Using External ssd instead of mac os ssd
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Amazon Reviews") \
    .master("local[4]") \
    .config("spark.driver.memory", "3g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.local.dir", "/Volumes/T7 Shield/spark-temp") \
    .getOrCreate()

26/05/29 21:05:30 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


In [72]:
spark.sparkContext.uiWebUrl

'http://192.0.0.2:4040'

In [74]:
reviews = spark.read.parquet("/Volumes/T7 Shield/data-project/parquet/reviews.parquet")
metadata = spark.read.parquet("/Volumes/T7 Shield/data-project/parquet/metadata.parquet")

In [75]:
reviews.printSchema()
metadata.printSchema()

reviews.show(5, vertical=True)
metadata.show(5, vertical=True)

reviews.groupBy("rating").show()

metadata.select(
    F.avg("price"),
    F.max("price"),
    F.min("price")
).show()

root
 |-- rating: double (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)

root
 |-- parent_asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- rating_number: long (nullable = true)
 |-- store: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- price: double (nullable = true)

-RECORD 0---------------------------------
 rating            | 5.0                  
 title             | Crazy comfy!         
 text              | Not gonna lie- th... 
 parent_asin       | B0BGFR76CF           
 user_id           | AFKZENTNBQ7A7V7UX... 
 timestamp         | 16773210

AttributeError: 'GroupedData' object has no attribute 'show'

In [78]:
spark.catalog.clearCache()

In [79]:

spark.stop()

In [61]:
# LDA (Latent Dirichlet Allocation)
reviews_sample = reviews.select("text") \
    .filter("text is not null") \
    .limit(100000)

In [63]:
# clean text
from pyspark.sql.functions import *

reviews_sample = reviews.select("text") \
    .filter(col("text").isNotNull())

reviews_sample = reviews_sample.withColumn(
    "text",
    lower(col("text"))
)

reviews_sample = reviews_sample.withColumn(
    "text",
    regexp_replace("text", "[^a-zA-Z ]", "")
)

In [64]:
# tokanize 
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="text",
    outputCol="words"
)

tokenized = tokenizer.transform(reviews_sample)

In [66]:
# remove stop words
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered_words"
)

filtered = remover.transform(tokenized)

In [67]:
# convert words to features
from pyspark.ml.feature import CountVectorizer

cv = CountVectorizer(
    inputCol="filtered_words",
    outputCol="features",
    vocabSize=5000,
    minDF=5
)

cv_model = cv.fit(filtered)

vectorized = cv_model.transform(filtered)

In [68]:
# train LDA
from pyspark.ml.clustering import LDA

lda = LDA(
    k=5,
    maxIter=10,
    featuresCol="features",
    seed=42
)

lda_model = lda.fit(vectorized)

26/05/29 20:55:18 WARN MemoryStore: Not enough space to cache rdd_312_1 in memory! (computed 47.7 MiB so far)
26/05/29 20:55:18 WARN BlockManager: Persisting block rdd_312_1 to disk instead.
26/05/29 20:55:20 WARN MemoryStore: Not enough space to cache rdd_312_6 in memory! (computed 54.4 MiB so far)
26/05/29 20:55:20 WARN BlockManager: Persisting block rdd_312_6 to disk instead.
26/05/29 20:55:20 WARN MemoryStore: Not enough space to cache rdd_312_0 in memory! (computed 45.8 MiB so far)
26/05/29 20:55:20 WARN BlockManager: Persisting block rdd_312_0 to disk instead.
26/05/29 20:55:24 WARN MemoryStore: Not enough space to cache rdd_312_2 in memory! (computed 64.6 MiB so far)
26/05/29 20:55:24 WARN BlockManager: Persisting block rdd_312_2 to disk instead.
26/05/29 20:55:24 WARN MemoryStore: Not enough space to cache rdd_312_3 in memory! (computed 73.1 MiB so far)
26/05/29 20:55:24 WARN BlockManager: Persisting block rdd_312_3 to disk instead.
26/05/29 20:55:29 WARN MemoryStore: Not enoug

Py4JJavaError: An error occurred while calling o773.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 7 in stage 121.0 failed 1 times, most recent failure: Lost task 7.0 in stage 121.0 (TID 581) (192.0.0.2 executor driver): java.io.IOException: No space left on device
	at java.base/sun.nio.ch.FileDispatcherImpl.write0(Native Method)
	at java.base/sun.nio.ch.FileDispatcherImpl.write(FileDispatcherImpl.java:62)
	at java.base/sun.nio.ch.IOUtil.writeFromNativeBuffer(IOUtil.java:132)
	at java.base/sun.nio.ch.IOUtil.write(IOUtil.java:97)
	at java.base/sun.nio.ch.IOUtil.write(IOUtil.java:67)
	at java.base/sun.nio.ch.FileChannelImpl.write(FileChannelImpl.java:288)
	at org.apache.spark.storage.CountingWritableChannel.write(DiskStore.scala:369)
	at java.base/java.nio.channels.Channels.writeFullyImpl(Channels.java:74)
	at java.base/java.nio.channels.Channels.writeFully(Channels.java:96)
	at java.base/java.nio.channels.Channels$1.write(Channels.java:171)
	at java.base/

In [ ]:
# view topics 
topics = lda_model.describeTopics(10)

topics.show(truncate=False)

In [ ]:
# convert to words
vocab = cv_model.vocabulary

topics_words = lda_model.describeTopics(10)

for topic in topics_words.collect():
    print(f"\nTopic {topic['topic']}")

    words = [vocab[idx] for idx in topic['termIndices']]

    print(words)